# Сянхэ: подготовка L0 из длинного датасета

Отдельный вариант `process_raw_data_China_Xianghe.ipynb` для `raw/Beijing/long dataset`.
Основные CSV имеют **те же названия колонок, порядок, формулы и единицы**, что и
таблицы короткого массива. Они сохраняются отдельно, в `processed/Beijing_long`.
Исходные ноутбуки и `processed/Beijing` этим ноутбуком не перезаписываются.

Дополнительные данные сохраняются в подпапках `meteo` и `radiation`, по одному CSV
на высоту. Первая колонка всех таблиц — `datetime_utc`, остальные — переменные.
Время переводится из UTC+8 в UTC так же, как в исходном ноутбуке. Данные не
агрегируются, не интерполируются и не обрезаются по датам в имени папки.
Повторяющиеся копии одной записи удаляются при полном совпадении значений.
Пустая служебная строка EddyPro `not_enough_data` удаляется только при наличии
измерений с той же меткой времени; самостоятельные пустые строки сохраняются.
Конфликтующие измерения с одной меткой времени останавливают обработку.


In [ ]:
from pathlib import Path
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import display

from l0_data import CHINA_LONG_ROOT, PROCESSED_ROOT
from china_long_data import (
    discover, load_flux, load_meteorology, write_frames,
    export_radiation, write_metadata,
)

city = 'Xianghe'
raw_root = CHINA_LONG_ROOT
export_dir = PROCESSED_ROOT / "Beijing_long"

# У радиации Сянхэ около 24 млн строк с шагом 3 с; запись идёт последовательно.
# При повторном запуске этот ноутбук пересоздаёт только файлы выбранной точки.
sources = []
outputs = []


## 1. Турбулентные потоки

Доступные высоты определяются по реальным папкам, включая `80M` в Пекине.
Для Пекина используются суточные `EP-Summary.txt`, для Сянхэ — почасовые TOA5
`Flux_Notes.dat`. Формулы `Qh`, `Qle`, `Qtau` и стандартных отклонений сохранены
из соответствующего ноутбука короткого массива. Компоненты ветра не поворачиваются.
Нулевые по размеру исходные файлы не содержат наблюдений и включаются в список
`empty_source_files` в метаданных; временные пробелы из-за них не заполняются.


In [ ]:
df4levels, flux_sources = load_flux(city, raw_root)
sources.extend(flux_sources)
flux_summary = pd.DataFrame([
    {"level": level, "rows": len(frame), "first_time_utc": frame.index.min(),
     "last_time_utc": frame.index.max(), "columns": list(frame.columns)}
    for level, frame in df4levels.items()
])
display(flux_summary)
outputs.extend(write_frames(city, df4levels, export_dir, kind="flux"))


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for level, frame in df4levels.items():
    # Прореживание используется только для отображения; CSV содержит все строки.
    view = frame.iloc[::max(1, len(frame) // 2500)]
    axes[0].plot(view.index, view["Qh"], label=level, linewidth=0.7)
    axes[1].plot(view.index, view["Wind"], label=level, linewidth=0.7)
axes[0].set_ylabel("Qh, W/m²")
axes[1].set_ylabel("Wind, m/s")
axes[1].set_xlabel("UTC")
for ax in axes:
    ax.legend(ncol=4)
    ax.grid(alpha=0.25)
fig.suptitle(city + ": длинный датасет, исходное временное разрешение")
fig.tight_layout()
plt.show()


## 2. Дополнительная метеорология

В TXT повторяются блоки высот, за которыми следуют `date=` и `hour=`.
Временная метка соответствует **концу напечатанного интервала**, включая переход
через полночь. Пекин: 30 минут, 15 высот; Сянхэ: 60 минут, 7 высот.
`Tair` остаётся в °C, `RHair` — в %, направления — в градусах.
Скорости двух приборов сохраняются отдельно: `Wind_nw`/`Wind_se` для Пекина,
`Wind_ne`/`Wind_sw` для Сянхэ. Выбор одного прибора и усреднение не выполняются.
Документированный маркер отсутствия данных `999` сохраняется как пропуск.

Верхний уровень Сянхэ назван `100m` согласно данным; в сопроводительном описании
он указан как 102 м. Температура отдельной метеосистемы не подменяет температуру
в таблице потоков Пекина, которая, как и в коротком массиве, выражена в K.


In [ ]:
meteo4levels, meteo_sources = load_meteorology(city, raw_root)
sources.extend(meteo_sources)
display(pd.DataFrame([
    {"level": level, "rows": len(frame), "first_time_utc": frame.index.min(),
     "last_time_utc": frame.index.max()}
    for level, frame in meteo4levels.items()
]))
outputs.extend(write_frames(city, meteo4levels, export_dir, kind="meteorology"))


## 3. Радиация

Пекин: 47/140/280 м, готовые 30-минутные записи. Читаются только текстовые
`TOA5_*.dat`; соседние двоичные TOB3 — исходные копии тех же выгрузок.
Сянхэ: 32/80 м, записи каждые 3 секунды; обработка и запись выполняются по файлам,
без загрузки всего периода в оперативную память и без агрегирования.

`SWdown`, `SWup`, `LWdown`, `LWup`, `Rn` — W/m²; `T_radiometer` — °C;
`Albedo` Сянхэ — безразмерное отношение. `LWUpperCo` и `LWLowerCo` уже содержат
температурную поправку прибора; она повторно не применяется.
Соответствие полей CNR4 подтверждается
[руководством Campbell Scientific](https://s.campbellsci.com/documents/es/manuals/874%20-%20cnr4.pdf).
Отрицательные ночные значения и прочие исходные значения не фильтруются.


In [ ]:
radiation_outputs, radiation_sources = export_radiation(city, export_dir, raw_root)
outputs.extend(radiation_outputs)
sources.extend(radiation_sources)
display(pd.DataFrame(radiation_outputs).drop(columns=["columns"]))


## 4. Итоговая диагностика и метаданные

Сохраняются фактическое покрытие, число строк и исходных файлов, единицы каждого
вида измерений, соответствие радиационных полей и SHA-256 выходных CSV.
Количество пропусков и пропуски во времени не исправляются автоматически.


In [ ]:
summary = write_metadata(city, outputs, sources, export_dir, raw_root)
display(summary)
print(f"Исходных файлов: {len(sources)}; выходных таблиц: {len(outputs)}")
print(f"Результаты: {export_dir}")
